# Извлечение текста из PDF через мультимодальную LLM

Пайплайн: **PDF → PNG → предобработка изображения → отправка страниц в LLM → сохранение результата в Markdown**.

Ноутбук рассчитан на работу в Jupyter Notebook / JupyterHub.

## 1. Установка библиотек

Запустите эту ячейку только если нужные библиотеки ещё не установлены.

In [ ]:
# !pip install pymupdf openai opencv-python numpy

## 2. Импорты

In [ ]:
import base64
import time
from pathlib import Path

import cv2
import fitz
import numpy as np

from openai import OpenAI
from IPython.display import display, Image, Markdown

## 3. Настройки и подключение к модели

In [ ]:
# Вставьте ваши параметры подключения
BASE_URL = "https://ai-gateway.raisa.go.rshbank.ru/v1"
MODEL = "Qwen/Qwen3.6-35B-test/sovetnik"
API_KEY = "ВАШ_API_KEY"

# Рекомендуемый диапазон для документов: 300–400 DPI
IMAGE_DPI = 400

INPUT_FOLDER = Path("./input")
OUTPUT_FOLDER = Path("./output")
TEMP_FOLDER = Path("./temp_images")

INPUT_FOLDER.mkdir(exist_ok=True)
OUTPUT_FOLDER.mkdir(exist_ok=True)
TEMP_FOLDER.mkdir(exist_ok=True)

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    timeout=300.0,
)

# Проверка подключения
client.models.list()

## 4. Промт для извлечения текста

In [ ]:
PROMPT = """
Твоя задача — максимально точно извлечь весь текст с изображения документа.

Правила:
- Извлекай весь видимый текст.
- Ничего не сокращай.
- Ничего не пересказывай.
- Ничего не придумывай.
- Не исправляй текст документа.
- Сохраняй числа, даты, ФИО и реквизиты точно как в документе.
- Сохраняй исходный порядок текста.
- Сохраняй заголовки.
- Таблицы представляй в Markdown с сохранением строк и столбцов.
- Если фрагмент невозможно уверенно прочитать, напиши [неразборчиво].
- Не добавляй комментарии и пояснения от себя.

В ответе должен быть только текст документа.
"""

## 5. PDF → PNG

In [ ]:
def pdf_to_images(pdf_path: Path, dpi=400) -> list[Path]:
    """Конвертирует каждую страницу PDF в PNG."""
    doc = fitz.open(pdf_path)

    folder = TEMP_FOLDER / pdf_path.stem
    folder.mkdir(parents=True, exist_ok=True)

    paths = []

    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        path = folder / f"page_{i:04d}.png"
        pix.save(str(path))
        paths.append(path)

    doc.close()
    return paths

## 6. Исправление небольшого перекоса (deskew)

Функция исправляет небольшой наклон страницы. Повороты на 90/180/270° она намеренно не выполняет.

In [ ]:
def deskew(image: np.ndarray) -> np.ndarray:
    binary = cv2.threshold(
        image,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )[1]

    coords = np.column_stack(np.where(binary > 0))

    if len(coords) == 0:
        return image

    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    # Исправляем только небольшой перекос.
    if abs(angle) > 10 or abs(angle) < 0.1:
        return image

    h, w = image.shape[:2]
    center = (w // 2, h // 2)

    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

    return cv2.warpAffine(
        image,
        matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE,
    )

## 7. Предобработка изображения

Применяется щадящая цепочка: **grayscale → denoise → CLAHE → sharpen → deskew**.

In [ ]:
def preprocess_image(image_path: Path) -> Path:
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Не удалось открыть изображение: {image_path}")

    # 1. Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 2. Лёгкое шумоподавление
    gray = cv2.fastNlMeansDenoising(gray, None, h=5)

    # 3. Локальное повышение контраста
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8),
    )
    enhanced = clahe.apply(gray)

    # 4. Лёгкое повышение резкости
    blurred = cv2.GaussianBlur(enhanced, (0, 0), 1.0)
    sharpened = cv2.addWeighted(enhanced, 1.5, blurred, -0.5, 0)

    # 5. Исправление небольшого перекоса
    result = deskew(sharpened)

    output_path = image_path.parent / f"{image_path.stem}_processed.png"
    cv2.imwrite(str(output_path), result)

    return output_path

## 8. PNG → Base64

In [ ]:
def image_to_base64(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

## 9. Отправка одной страницы в модель

In [ ]:
def extract_page(image_path: Path, max_retries=3) -> str:
    image_base64 = image_to_base64(image_path)

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": PROMPT,
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": "data:image/png;base64," + image_base64
                                },
                            }
                        ],
                    },
                ],
                temperature=0,
                extra_body={
                    "top_k": 20,
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    },
                },
            )

            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Ошибка запроса, попытка {attempt}/{max_retries}: {e}")

            if attempt == max_retries:
                raise

            time.sleep(5)

## 10. Выбор PDF для теста

Положите PDF в папку `input` и укажите имя файла ниже.

In [ ]:
pdf_path = INPUT_FOLDER / "1_цель.pdf"

image_paths = pdf_to_images(
    pdf_path,
    dpi=IMAGE_DPI,
)

print(f"Страниц: {len(image_paths)}")

## 11. Сравнение исходной и обработанной страницы

In [ ]:
# Номер страницы для визуального теста, начиная с 1
TEST_PAGE = 1

original_path = image_paths[TEST_PAGE - 1]
processed_path = preprocess_image(original_path)

print("Исходная страница:")
display(Image(filename=str(original_path)))

print("После предобработки:")
display(Image(filename=str(processed_path)))

## 12. Тест извлечения текста с одной страницы

In [ ]:
text = extract_page(processed_path)
display(Markdown(text))

## 13. Сравнение LLM на исходном и обработанном изображении

Эта ячейка помогает проверить, действительно ли предобработка улучшает извлечение.

In [ ]:
text_original = extract_page(original_path)
text_processed = extract_page(processed_path)

display(Markdown("## Исходное изображение\n\n" + text_original))
display(Markdown("## После предобработки\n\n" + text_processed))

## 14. Обработка всего выбранного PDF

In [ ]:
all_text = []

for i, image_path in enumerate(image_paths, start=1):
    print(f"Обработка страницы {i}/{len(image_paths)}")

    processed = preprocess_image(image_path)
    page_text = extract_page(processed)

    all_text.append(
        f"# Страница {i}\n\n{page_text}"
    )

    time.sleep(1)

full_text = "\n\n---\n\n".join(all_text)

print("Готово")

## 15. Просмотр итогового текста

In [ ]:
display(Markdown(full_text))

## 16. Сохранение результата в Markdown

In [ ]:
output_path = OUTPUT_FOLDER / f"{pdf_path.stem}.md"

output_path.write_text(
    full_text,
    encoding="utf-8",
)

print(f"Сохранено: {output_path}")

## 17. Опционально: обработка всех PDF из папки `input`

Эту ячейку запускайте, когда убедитесь, что настройки предобработки и промт дают нужное качество.

In [ ]:
def process_pdf(pdf_path: Path):
    print(f"\n{'=' * 70}")
    print(f"PDF: {pdf_path.name}")
    print(f"{'=' * 70}")

    images = pdf_to_images(pdf_path, dpi=IMAGE_DPI)
    parts = []

    for i, image_path in enumerate(images, start=1):
        print(f"Страница {i}/{len(images)}")

        processed = preprocess_image(image_path)
        page_text = extract_page(processed)

        parts.append(f"# Страница {i}\n\n{page_text}")
        time.sleep(1)

    result = "\n\n---\n\n".join(parts)
    output_path = OUTPUT_FOLDER / f"{pdf_path.stem}.md"
    output_path.write_text(result, encoding="utf-8")

    print(f"Сохранено: {output_path}")
    return output_path


pdf_files = sorted(INPUT_FOLDER.glob("*.pdf"))
print(f"Найдено PDF: {len(pdf_files)}")

for file in pdf_files:
    try:
        process_pdf(file)
    except Exception as e:
        print(f"Ошибка {file.name}: {e}")

## Примечания

- Для документов обычно нет смысла сразу повышать разрешение выше 400 DPI: запрос становится существенно тяжелее.
- Предобработка не гарантирует улучшение каждого документа. Поэтому в ноутбуке предусмотрено сравнение исходного и обработанного изображения.
- Бинаризация намеренно не включена в основной пайплайн: она может удалить тонкие линии, слабый текст, подписи и печати.
- API-ключ лучше не хранить в ноутбуке, если файл будет кому-либо передаваться.